# Trabalho Grau B — Classificação de Cenas Sonoras

**Disciplina:** Inteligência Artificial e Aprendizado de Máquina (2026/1)  
**Professor:** Gabriel de Oliveira Ramos  
**Relatório Parcial — 17/05/2025**

---

## 1. Descrição do Problema

O trabalho consiste em identificar automaticamente quais sons estão presentes em uma gravação de áudio. Diferente de um problema de classificação simples, aqui um mesmo áudio pode conter vários sons acontecendo ao mesmo tempo — alguém usando um secador de cabelo enquanto uma campainha toca, por exemplo. Isso caracteriza um problema de **classificação multi-label**, onde a saída do modelo é um conjunto de classes e não apenas uma.

O dataset escolhido é o do desafio **DCASE 2025** (Detection and Classification of Acoustic Scenes and Events) [1], um benchmark internacional para tarefas de análise de cenas acústicas.

A conexão com o tema de classificação de imagens vem da técnica de pré-processamento que vamos usar: converter o áudio em um **espectrograma de Mel**, que é essencialmente uma imagem 2D representando energia por frequência ao longo do tempo. Com isso, o problema de classificar sons vira um problema de classificar imagens, e CNNs se aplicam diretamente. Essa abordagem é bastante consolidada na literatura — Hershey et al. [2] mostraram em 2017 que CNNs treinadas em espectrogramas superam métodos clássicos de análise de áudio em várias tarefas.

## 2. Dataset

### 2.1 Visão Geral

O dataset contém **2.290 arquivos de áudio** distribuídos em **18 classes** de sons. Das 18 classes disponíveis, vamos trabalhar com **5**, selecionadas com base na distribuição de amostras — o critério foi escolher classes com quantidade de amostras parecida entre si para evitar desbalanceamento severo no treinamento.

Alguns exemplos de categorias presentes no dataset: campainha, secador de cabelo e ventilador.

### 2.2 Classes Selecionadas

*(🔲 a ser preenchido pelo colega — quais 5 classes foram escolhidas, quantas amostras cada uma tem e justificativa da escolha)*

### 2.3 Análise Exploratória

*(🔲 a ser preenchido pelo colega — distribuição de amostras, duração média dos áudios, visualização de algum espectrograma de exemplo)*

## 3. Pré-processamento

O pipeline vai de arquivo `.wav` até um tensor pronto para a CNN, seguindo as etapas: carregar o áudio → gerar o espectrograma de Mel → normalizar → dividir os dados → aplicar data augmentation durante o treino.

### 3.1 Carregamento do Áudio

Todos os arquivos serão carregados em mono com taxa de amostragem de **22.050 Hz**, que é o padrão da biblioteca librosa [3] e cobre frequências de até ~11 kHz — suficiente para os sons que vamos classificar. Para garantir que todos os inputs tenham o mesmo tamanho, fixamos a duração em **4 segundos**: áudios mais longos são cortados e mais curtos recebem zero-padding no final.

### 3.2 Espectrograma de Mel

Optamos pelo espectrograma de Mel como representação do áudio. A alternativa mais direta seria dar o áudio bruto para a rede, mas isso exigiria que ela aprendesse sozinha todas as relações de frequência a partir de amostras brutas — o que demanda muito mais dados e tempo de treino. MFCCs também são uma opção comum, mas descartam informação de fase e energia que pode ser útil para distinguir sons parecidos.

O espectrograma de Mel é uma imagem 2D (frequências na escala Mel × tempo) que já entrega para a CNN as features espectrais mais relevantes. A escala Mel aproxima a percepção humana de frequência, dando mais resolução às frequências baixas. Essa escolha é bem suportada pela literatura — Salamon e Bello [4] mostraram que CNNs aplicadas a espectrogramas de Mel superam abordagens baseadas em features manuais para classificação de sons urbanos.

Os parâmetros que vamos usar são `n_mels=128`, `n_fft=2048` e `hop_length=512`, que são valores bastante padrão para esse tipo de tarefa. Convertemos a amplitude para dB com `power_to_db` para comprimir a escala logaritmicamente — sem isso, diferenças de volume entre amostras dificultariam o treinamento. O espectrograma gerado (~128×173) será redimensionado para **128×128** para uniformizar o input.

### 3.3 Normalização e Divisão dos Dados

Após gerar o espectrograma, normalizamos os valores para [0, 1] via min-max por amostra. Isso evita que amostras com amplitude naturalmente mais alta dominem o gradiente durante o treino.

Os dados serão divididos em **70% treino, 15% validação e 15% teste**. A validação serve para monitorar overfitting e ajustar hiperparâmetros; o conjunto de teste fica separado e só é usado para reportar os resultados finais.

### 3.4 Data Augmentation

Com ~2.290 amostras no total o dataset é pequeno para treinar CNNs sem overfitting. Vamos usar data augmentation para aumentar artificialmente a diversidade do conjunto de treino.

No sinal de áudio, antes de gerar o espectrograma, aplicamos **time shifting** (deslocar o áudio no tempo em até ±20%) e **adição de ruído gaussiano** suave. Diretamente no espectrograma, vamos usar **SpecAugment** [5] — mascarar faixas horizontais de frequência e faixas verticais de tempo aleatoriamente. SpecAugment foi proposto por Park et al. em 2019 para reconhecimento de fala e funcionou bem em tarefas de classificação de áudio em geral, inclusive com datasets pequenos.

## 4. Arquiteturas das Redes

Vamos implementar duas CNNs: uma treinada do zero e outra com transfer learning. Ambas recebem espectrogramas de Mel com shape **(128, 128, 1)** e produzem um vetor de 5 valores — um por classe — com **ativação sigmoid** na saída. Sigmoid é a escolha certa aqui porque o problema é multi-label: cada saída é independente e pode assumir qualquer valor entre 0 e 1, sem a restrição de somar 1 que o softmax impõe. O limiar de 0.5 vai definir quais classes são consideradas presentes.

### 4.1 CNN do Zero

A arquitetura segue o padrão clássico de blocos convolucionais (Conv2D → BatchNorm → MaxPool) seguidos de camadas densas, similar ao descrito em trabalhos de referência para classificação de sons com espectrogramas [2, 4].

A ideia é que as primeiras camadas convolucionais capturem padrões simples no espectrograma — bordas, variações bruscas de frequência — e as camadas mais profundas combinem esses padrões em representações mais abstratas, como o envelope temporal característico de um secador de cabelo ou os harmônicos de uma campainha.

Usamos **BatchNormalization** depois de cada convolução para estabilizar o treinamento [6] e **GlobalAveragePooling2D** no lugar de Flatten antes das camadas densas — isso reduz bastante o número de parâmetros e ajuda a controlar o overfitting. Um **Dropout de 0.5** na camada densa serve como regularização adicional.

A configuração base para os experimentos será:
- 3 blocos convolucionais com 32 → 64 → 128 filtros
- 1 camada densa com 256 neurônios
- Ativação ReLU, Binary Cross-Entropy, otimizador Adam

### 4.2 Transfer Learning com EfficientNetB0

Para a segunda rede, optamos pelo **EfficientNetB0** [7] como backbone pré-treinado no ImageNet. A principal razão para não usar ResNet50 ou VGG16 é o tamanho: VGG16 tem ~138M de parâmetros e ResNet50 ~25M — ambos superdimensionados para um dataset de pouco mais de 2 mil amostras. EfficientNetB0 tem ~5.3M parâmetros e ainda assim alcança accuracy maior no ImageNet que o ResNet50, por escalar largura, profundidade e resolução de forma balanceada.

A transferência de um modelo treinado em imagens naturais para espectrogramas faz sentido porque as primeiras camadas de uma CNN aprendem detectores genéricos — bordas, texturas, gradientes locais — que são igualmente informativos em espectrogramas (bordas verticais correspondem a transientes sonoros, horizontais a tons estacionários).

A estratégia de treinamento terá duas fases: primeiro congelamos o backbone e treinamos só a cabeça de classificação por algumas épocas, depois descongelamos as últimas camadas para um fine-tuning com taxa de aprendizado bem menor. Isso evita destruir os pesos pré-treinados nas primeiras épocas, quando os gradientes da cabeça ainda são grandes.

O EfficientNetB0 espera entrada RGB (3 canais). Como nosso espectrograma tem apenas 1 canal, vamos replicá-lo 3 vezes antes de passar pelo backbone.

### 4.3 Interpretação da Saída

As redes produzem um vetor de probabilidades por classe. Para converter em predição final, aplicamos um limiar de 0.5 e mapeamos para os nomes das classes. Por exemplo, um output `[0.92, 0.08, 0.87, 0.13, 0.04]` resultaria em `['campainha', 'ventilador']`.

## 5. Planejamento dos Experimentos

Para cada rede vamos rodar pelo menos 12 experimentos variando hiperparâmetros. A ideia é partir de uma configuração baseline e ir mudando um hiperparâmetro por vez para entender o impacto de cada um, e depois testar algumas combinações.

Os hiperparâmetros que vamos variar são:
- **Número de camadas convolucionais** (ou layers descongeladas no fine-tuning): 3 variações
- **Neurônios na camada densa**: 128, 256 ou 512
- **Função de ativação**: ReLU ou ELU
- **Função de perda**: Binary Cross-Entropy ou Focal Loss
- **Otimizador**: Adam ou SGD com momentum

A métrica principal será o **F1-Score macro**, que é mais adequada para multi-label com possível desbalanceamento entre classes do que accuracy simples. Também vamos reportar a val_loss para acompanhar overfitting.

A justificativa para variar a função de ativação é que ELU pode ter vantagem sobre ReLU em redes mais profundas por não sofrer com o problema de neurônios mortos [8]. Para a função de perda, Focal Loss [9] penaliza mais os exemplos difíceis — pode ser útil se algumas classes forem mais raras ou parecidas entre si. Para o otimizador, Adam converge mais rápido mas SGD com momentum às vezes generaliza melhor.

### 5.1 Experimentos — CNN do Zero

| Exp | Blocos Conv | Neurônios Dense | Ativação | Loss | Otimizador | F1 (val) | Val Loss |
|-----|-------------|-----------------|----------|------|------------|----------|----------|
| 1 | 2 | 256 | ReLU | BCE | Adam | — | — |
| 2 *(baseline)* | 3 | 256 | ReLU | BCE | Adam | — | — |
| 3 | 4 | 256 | ReLU | BCE | Adam | — | — |
| 4 | 3 | 128 | ReLU | BCE | Adam | — | — |
| 5 | 3 | 512 | ReLU | BCE | Adam | — | — |
| 6 | 3 | 256 | ELU | BCE | Adam | — | — |
| 7 | 3 | 256 | ReLU | Focal | Adam | — | — |
| 8 | 3 | 256 | ReLU | BCE | SGD | — | — |
| 9 | 2 | 128 | ELU | BCE | Adam | — | — |
| 10 | 4 | 512 | ReLU | BCE | Adam | — | — |
| 11 | 3 | 256 | ELU | Focal | Adam | — | — |
| 12 | 4 | 256 | ELU | Focal | SGD | — | — |

### 5.2 Experimentos — Transfer Learning (EfficientNetB0)

| Exp | Layers descongeladas | Neurônios Dense | Ativação | Loss | Otimizador | F1 (val) | Val Loss |
|-----|---------------------|-----------------|----------|------|------------|----------|----------|
| 1 *(baseline)* | 0 (frozen) | 256 | ReLU | BCE | Adam | — | — |
| 2 | 20 | 256 | ReLU | BCE | Adam | — | — |
| 3 | 50 | 256 | ReLU | BCE | Adam | — | — |
| 4 | 0 | 128 | ReLU | BCE | Adam | — | — |
| 5 | 0 | 512 | ReLU | BCE | Adam | — | — |
| 6 | 0 | 256 | ELU | BCE | Adam | — | — |
| 7 | 0 | 256 | ReLU | Focal | Adam | — | — |
| 8 | 0 | 256 | ReLU | BCE | SGD | — | — |
| 9 | 20 | 256 | ELU | BCE | Adam | — | — |
| 10 | 50 | 512 | ReLU | BCE | Adam | — | — |
| 11 | 20 | 256 | ELU | Focal | Adam | — | — |
| 12 | 50 | 256 | ELU | Focal | SGD | — | — |

## 6. Configurações de Treinamento

Todos os experimentos vão usar as mesmas configurações fixas para garantir comparabilidade: máximo de **50 épocas**, **batch size 32** e semente aleatória 42.

Vamos usar três callbacks durante o treino. O **EarlyStopping** com patience=10 interrompe o treino se a val_loss não melhorar por 10 épocas seguidas, restaurando os pesos da melhor época — isso evita overfitting sem precisar definir um número fixo de épocas por experimento. O **ModelCheckpoint** salva automaticamente o modelo com menor val_loss. O **ReduceLROnPlateau** reduz a learning rate pela metade se a val_loss estagnar por 5 épocas, o que ajuda a encontrar mínimos mais finos nas etapas finais do treino.

## Referências

[1] DCASE Community. *DCASE 2025 Challenge — Task: Spatial Semantic Segmentation of Sound Scenes*. Disponível em: https://dcase.community/challenge2025/

[2] HERSHEY, S. et al. CNN architectures for large-scale audio classification. In: *IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP)*, 2017.

[3] McFEE, B. et al. librosa: Audio and music signal analysis in Python. In: *Proceedings of the 14th Python in Science Conference*, 2015.

[4] SALAMON, J.; BELLO, J. P. Deep convolutional neural networks and data augmentation for environmental sound classification. *IEEE Signal Processing Letters*, v. 24, n. 3, p. 279–283, 2017.

[5] PARK, D. S. et al. SpecAugment: A simple data augmentation method for automatic speech recognition. In: *Interspeech*, 2019.

[6] IOFFE, S.; SZEGEDY, C. Batch normalization: Accelerating deep network training by reducing internal covariate shift. In: *International Conference on Machine Learning (ICML)*, 2015.

[7] TAN, M.; LE, Q. V. EfficientNet: Rethinking model scaling for convolutional neural networks. In: *International Conference on Machine Learning (ICML)*, 2019.

[8] CLEVERT, D.; UNTERTHINER, T.; HOCHREITER, S. Fast and accurate deep network learning by exponential linear units (ELUs). In: *International Conference on Learning Representations (ICLR)*, 2016.

[9] LIN, T. et al. Focal loss for dense object detection. In: *IEEE International Conference on Computer Vision (ICCV)*, 2017.